# Silver — Partner Events

Normalizes 6–7 heterogeneous operator feeds into one canonical schema and merges into `silver.partner_events`.

**Key design choices**
- One normalizer per operator. 
- MERGE key is `(operator_code, partner_txn_id, business_date)`. 
- Every row carries both `business_date` (when it happened) and
  `ingestion_date` (when we learned).

**Inputs**: CSV files in `Files/Sample_Data/Partner/<operator>/<ingestion_date>/`
**Output**: Delta table `silver.partner_events`


In [1]:
import datetime as dt

# This cell is parameterized 
ingestion_date = dt.date.today().isoformat()
raw_root = "Files/Sample_Data/Partner"
silver_table = "silver_partner_events"

print(f"ingestion_date = {ingestion_date}")
print(f"raw_root = {raw_root}")
print(f"silver_table = {silver_table}")

StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 3, Finished, Available, Finished, False)

ingestion_date = 2026-05-27
raw_root = Files/Sample_Data/Partner
silver_table = silver_partner_events


## Defining Canonical schema


In [2]:
from pyspark.sql import DataFrame, functions as F, types as T, Window as W
from delta.tables import DeltaTable

PARTNER_EVENT_SCHEMA = T.StructType([
    T.StructField("operator_code",T.StringType(),False),
    T.StructField("partner_txn_id",T.StringType(),False),
    T.StructField("partner_account_id",T.StringType(),True),
    T.StructField("txn_type",T.StringType(),False),
    T.StructField("plan_code",T.StringType(),True),
    T.StructField("amount",T.DecimalType(18, 4),False),
    T.StructField("currency",T.StringType(),False),
    T.StructField("txn_ts_utc",T.TimestampType(),False),
    T.StructField("business_date",T.DateType(),False),
    T.StructField("ingestion_date",T.DateType(),False),
    T.StructField("source_file",T.StringType(),False),
    T.StructField("_ingested_at",T.TimestampType(),False),
])

CANONICAL_TXN_TYPES = {"SUB_OK", "RECUR_OK", "RECUR_FAIL", "REFUND", "CANCEL"}


StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 4, Finished, Available, Finished, False)

## Registry

We define each operators's normalizer below, then populate the registry in one place. 
Adding new operator -> a new function and a new dict entry.


## Normalizers per Operator

In [4]:
def normalize_telco_a(raw: DataFrame, source_file: str, ingestion_date: str) -> DataFrame:
    
    type_map = F.create_map([F.lit(x) for pair in [
        ("SUB_OK",     "SUB_OK"),
        ("RECUR_OK",   "RECUR_OK"),
        ("RECUR_FAIL", "RECUR_FAIL"),
        ("REFUND",     "REFUND"),
    ] for x in pair])

    df = (raw
        # convert per-row local time to UTC
        .withColumn("txn_ts_utc",
            F.to_utc_timestamp(F.to_timestamp(F.col("ts_local")), F.col("tz")))
        .withColumn("operator_code",F.lit("telco_a"))
        .withColumn("partner_txn_id",F.col("op_txn"))
        .withColumn("partner_account_id",F.col("msisdn").cast("string"))
        .withColumn("txn_type",type_map[F.col("type")])
        .withColumn("plan_code",F.col("plan"))
        .withColumn("amount",F.col("amt").cast(T.DecimalType(18, 4)))
        .withColumn("currency",F.col("ccy"))
        .withColumn("business_date", F.to_date(F.col("txn_ts_utc")))
        .withColumn("ingestion_date",F.lit(ingestion_date).cast(T.DateType()))
        .withColumn("source_file",F.lit(source_file))
        .withColumn("_ingested_at",F.current_timestamp())
    )
    return df.select(*[f.name for f in PARTNER_EVENT_SCHEMA.fields])


StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 6, Finished, Available, Finished, False)

In [5]:
def normalize_telco_b(raw: DataFrame, source_file: str, ingestion_date: str) -> DataFrame:
    
    type_map = F.create_map([F.lit(x) for pair in [
        ("subscription_success", "SUB_OK"),
        ("recursion_success",    "RECUR_OK"),
        ("recursion_failure",    "RECUR_FAIL"),
        ("refund",               "REFUND"),
        ("cancel",               "CANCEL"),
    ] for x in pair])

    df = (raw
        .withColumn("txn_ts_utc",
            F.to_utc_timestamp(F.to_timestamp(F.col("occurred_at")), F.col("timezone")))
        .withColumn("operator_code",F.lit("telco_b"))
        .withColumn("partner_txn_id",F.col("partner_reference"))
        .withColumn("partner_account_id", F.col("account").cast("string"))
        .withColumn("txn_type", type_map[F.col("event")])
        .withColumn("plan_code", F.col("product"))
        .withColumn("amount",F.col("value").cast(T.DecimalType(18, 4)))
        .withColumn("currency",F.col("curr"))
        .withColumn("business_date", F.to_date(F.col("txn_ts_utc")))
        .withColumn("ingestion_date",F.lit(ingestion_date).cast(T.DateType()))
        .withColumn("source_file",F.lit(source_file))
        .withColumn("_ingested_at",F.current_timestamp())
    )
    return df.select(*[f.name for f in PARTNER_EVENT_SCHEMA.fields])


StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 7, Finished, Available, Finished, False)

## Normalizers Dict

One place that lists which normalizer handles which operator code.

In [7]:
NORMALIZERS = {
    "telco_a": normalize_telco_a,
    "telco_b": normalize_telco_b
}

StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 9, Finished, Available, Finished, False)

## MERGE

The merge key — `(operator_code, partner_txn_id, business_date)`


In [8]:
def merge_into_silver(new_events: DataFrame, table_name: str):
    if not spark.catalog.tableExists(table_name):
        (new_events.write
            .format("delta")
            .partitionBy("business_date", "operator_code")
            .saveAsTable(table_name))
        print(f"Created {table_name} with {new_events.count()} rows")
        return

    target = DeltaTable.forName(spark, table_name)
    (target.alias("t").merge(
        new_events.alias("s"),
        condition=(
            "t.operator_code = s.operator_code "
            "AND t.partner_txn_id = s.partner_txn_id "
            "AND t.business_date = s.business_date"
        ))
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged into {table_name}")


StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 10, Finished, Available, Finished, False)

For each operator, find today's RAW files, normalize, dedupe within the file, merge into silver.


In [9]:
def build_slv_partner_evt(ingestion_date: str):
    for operator_code, normalizer in NORMALIZERS.items():
        raw_path = f"{raw_root}/{operator_code}/{ingestion_date}/"

        try:
            raw = (spark.read
                .option("header", "true")
                .option("inferSchema", "false")
                .csv(raw_path))
        except Exception as e:
            print(f"  {operator_code}: no files at {raw_path} — {type(e).__name__}")
            continue

        if not raw.first():
            print(f"  {operator_code}: empty input at {raw_path}")
            continue

        normalized = normalizer(raw, source_file=raw_path, ingestion_date=ingestion_date)

        # Dedupe within file
        w = W.partitionBy("operator_code", "partner_txn_id", "business_date") \
             .orderBy(F.col("_ingested_at").desc())
        normalized = (normalized
            .withColumn("_rn", F.row_number().over(w))
            .filter("_rn = 1")
            .drop("_rn"))
        
        merge_into_silver(normalized, silver_table)
        print(f"  {operator_code}: ingested {normalized.count()} rows")


StatementMeta(, bf01ebfb-14e2-4dc3-87ce-f75b83ce1853, 11, Finished, Available, Finished, False)

## Run

In [47]:
build_slv_partner_evt(ingestion_date)


StatementMeta(, 0f61afdc-3bef-447e-9038-56d5b5ccb7b0, 49, Finished, Available, Finished, False)

Merged into silver_partner_events
  telco_a: ingested 9 rows
Merged into silver_partner_events
  telco_b: ingested 3 rows


In [49]:
# display(spark.sql(f"SELECT * FROM {silver_table}"))

StatementMeta(, 0f61afdc-3bef-447e-9038-56d5b5ccb7b0, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 578730f2-fa96-4327-99ba-548651cb6d1c)